In [ ]:
import re
import json
from pathlib import Path
from collections import defaultdict

import spacy
from spacy.matcher import Matcher

MODEL_NAME = "en_core_web_sm"

try:
    nlp = spacy.load(MODEL_NAME)
except OSError:
    raise RuntimeError(f"spaCy model '{MODEL_NAME}' is not installed. ")

print("spaCy:", spacy.__version__)
print("NER model loaded:", MODEL_NAME)


spaCy: 3.8.14
NER model loaded: en_core_web_sm


In [ ]:
class InformationExtractor:
    EMAIL_RE = re.compile(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    )
    URL_RE = re.compile(r"https?://[^\s<>\"]+")

    AUDIENCE_PATTERNS = {
        "Students": [
            r"\bstudents?\b",
            r"\ball\s+students?\b",
            r"\bfinal[- ]year students?\b",
        ],
        "Faculty": [
            r"\bfacult(y|ies)\b",
            r"\bteachers?\b",
            r"\bcourse instructors?\b",
            r"\bprofessors?\b",
        ],
        "Staff": [
            r"\bstaff\b",
            r"\badministrative staff\b",
        ],
        "Applicants": [
            r"\bapplicants?\b",
            r"\bcandidates?\b",
        ],
        "Researchers": [
            r"\bresearchers?\b",
            r"\bscholars?\b",
        ],
    }

    ACTION_VERBS = {
        "submit", "apply", "register", "upload", "attend", "pay",
        "complete", "verify", "confirm", "report", "return", "provide",
        "notify", "contact", "fill", "file", "deposit", "collect",
        "present", "appear", "participate", "respond", "ensure",
        "maintain", "follow", "comply", "obtain", "renew", "renewal",
        "check", "approve", "forward", "send", "email"
    }

    OBLIGATION_WORDS = {
        "must", "shall", "required", "required to", "need to",
        "should", "requested to", "responsible for", "have to"
    }

    def __init__(self, nlp):
        self.nlp = nlp
        self.matcher = Matcher(nlp.vocab)
        self._setup_action_patterns()

    def _setup_action_patterns(self):
        patterns = [
            [
                {"LOWER": {"IN": ["must", "shall", "should"]}},
                {"POS": {"IN": ["ADV", "PART"]}, "OP": "*"},
                {"POS": "VERB"},
            ],
            [
                {"LOWER": {"IN": ["required", "requested"]}},
                {"LOWER": "to", "OP": "?"},
                {"POS": "VERB"},
            ],
            [
                {"LOWER": "are"},
                {"LOWER": {"IN": ["required", "requested"]}},
                {"LOWER": "to", "OP": "?"},
                {"POS": "VERB"},
            ],
            [
                {"LOWER": "is"},
                {"LOWER": {"IN": ["required", "requested"]}},
                {"LOWER": "to", "OP": "?"},
                {"POS": "VERB"},
            ],
        ]
        for i, pattern in enumerate(patterns):
            self.matcher.add(f"ACTION_RULE_{i}", [pattern])

    @staticmethod
    def _unique(values):
        return list(dict.fromkeys(v.strip() for v in values if v and v.strip()))

    def _extract_contacts(self, text):
        emails = self._unique(self.EMAIL_RE.findall(text))
        urls = self._unique([u.rstrip(".,;:)") for u in self.URL_RE.findall(text)])
        return emails, urls

    def _extract_audiences(self, sentence):
        found = []
        for label, patterns in self.AUDIENCE_PATTERNS.items():
            if any(re.search(p, sentence, re.I) for p in patterns):
                found.append(label)
        return found

    def _extract_action_spans(self, doc):
        actions = []
        seen = set()

        for _, start, end in self.matcher(doc):
            span = doc[start:end]
            sentence = span.sent.text.strip()

            if sentence and sentence not in seen:
                seen.add(sentence)
                actions.append({
                    "action": span.text.strip(),
                    "sentence": sentence
                })

        for sent in doc.sents:
            text = sent.text.strip()
            low = text.lower()

            has_obligation = any(
                re.search(r"\b" + re.escape(w) + r"\b", low)
                for w in self.OBLIGATION_WORDS
            )
            has_action_verb = any(
                re.search(r"\b" + re.escape(v) + r"\b", low)
                for v in self.ACTION_VERBS
            )

            if has_obligation and has_action_verb and text not in seen:
                seen.add(text)
                actions.append({
                    "action": text,
                    "sentence": text
                })

        return actions

    def _extract_relations(self, doc, actions):
        relations = []

        for action_item in actions:
            sentence = action_item["sentence"]
            sent_doc = self.nlp(sentence)

            ents = list(sent_doc.ents)
            audiences = self._extract_audiences(sentence)

            # ACTION -> AUDIENCE
            for audience in audiences:
                relations.append({
                    "type": "ACTION_TARGETS",
                    "source": action_item["action"],
                    "target": audience,
                    "evidence": sentence
                })

            # ACTION -> DATE/TIME
            for ent in ents:
                if ent.label_ in {"DATE", "TIME"}:
                    relations.append({
                        "type": "ACTION_HAS_DEADLINE_OR_TIME",
                        "source": action_item["action"],
                        "target": ent.text,
                        "label": ent.label_,
                        "evidence": sentence
                    })

            # ACTION -> ORGANIZATION
            for ent in ents:
                if ent.label_ == "ORG":
                    relations.append({
                        "type": "ACTION_INVOLVES_ORG",
                        "source": action_item["action"],
                        "target": ent.text,
                        "evidence": sentence
                    })

        # PERSON -> ORG when both appear in the same sentence.
        for sent in doc.sents:
            ents = list(sent.ents)
            people = [e.text for e in ents if e.label_ == "PERSON"]
            orgs = [e.text for e in ents if e.label_ == "ORG"]

            for person in people:
                for org in orgs:
                    relations.append({
                        "type": "PERSON_AFFILIATED_WITH_ORG",
                        "source": person,
                        "target": org,
                        "evidence": sent.text.strip()
                    })

        unique = []
        seen = set()
        for rel in relations:
            key = (
                rel["type"],
                rel["source"],
                rel["target"],
                rel["evidence"]
            )
            if key not in seen:
                seen.add(key)
                unique.append(rel)

        return unique

    def extract_entities(self, text):
        if not isinstance(text, str) or not text.strip():
            return {
                "entities": [],
                "deadlines_and_dates": [],
                "organizations": [],
                "persons": [],
                "times": [],
                "contact_emails": [],
                "urls": [],
                "actions": [],
                "relations": []
            }

        doc = self.nlp(text)
        emails, urls = self._extract_contacts(text)

        entities = []
        for ent in doc.ents:
            entities.append({
                "text": ent.text.strip(),
                "label": ent.label_,
                "start": ent.start_char,
                "end": ent.end_char
            })

        dates = self._unique([e.text for e in doc.ents if e.label_ == "DATE"])
        times = self._unique([e.text for e in doc.ents if e.label_ == "TIME"])
        organizations = self._unique([e.text for e in doc.ents if e.label_ == "ORG"])
        persons = self._unique([e.text for e in doc.ents if e.label_ == "PERSON"])

        actions = self._extract_action_spans(doc)
        relations = self._extract_relations(doc, actions)

        return {
            "entities": entities,
            "deadlines_and_dates": dates,
            "times": times,
            "contact_emails": emails,
            "urls": urls,
            "organizations": organizations,
            "persons": persons,
            "actions": actions,
            "relations": relations
        }


extractor = InformationExtractor(nlp)
print("NER + relationship extraction engine initialized.")


NER + relationship extraction engine initialized.


In [10]:
sample_notice_text = """Dear Parents and Students,
It has come to the notice of the undersigned that certain students/agents are circulating interest/registration forms and making arrangements for outstation trips, including trips to destinations such as Manali, while misrepresenting or giving an impression that such trips are organised, authorised, or endorsed by the University/School.
It is hereby clarified that the University/School does not organise, authorise, endorse, or entertain any such unofficial/private outstation trips for students. Any person or group arranging such trips is doing so without the authorisation of the University/School.
Students are strongly advised not to register for, make payments towards, or participate in such trips. Students are also advised not to join WhatsApp or other social-media groups created for such unauthorised trips, as doing so may expose their personal information and contact details to unknown persons or third parties.
The University/School shall not be responsible for any such unauthorised activities or for any consequences arising from participation in them.
Parents are requested to exercise due caution and rely exclusively on official communications issued through the authorised channels of the University/School. If parents or students come across any forms, messages, calls, social-media posts, WhatsApp groups, or other communications relating to such trips that claim or imply University/School involvement, they are requested to immediately report the same to the School authorities.
This communication is being issued in the interest of student safety, privacy, and security, and to prevent any misrepresentation of the University/School.
Thanks and Regards,
Dean
MPSTME, NMIMS"""

result = extractor.extract_entities(sample_notice_text)
print(json.dumps(result, indent=2))


{
  "entities": [
    {
      "text": "Dear Parents and Students",
      "label": "ORG",
      "start": 0,
      "end": 25
    },
    {
      "text": "Manali",
      "label": "PERSON",
      "start": 231,
      "end": 237
    },
    {
      "text": "the University/School",
      "label": "ORG",
      "start": 343,
      "end": 364
    },
    {
      "text": "the University/School",
      "label": "ORG",
      "start": 394,
      "end": 415
    },
    {
      "text": "the University/School",
      "label": "ORG",
      "start": 609,
      "end": 630
    },
    {
      "text": "WhatsApp",
      "label": "ORG",
      "start": 774,
      "end": 782
    },
    {
      "text": "third",
      "label": "ORDINAL",
      "start": 941,
      "end": 946
    },
    {
      "text": "The University/School",
      "label": "ORG",
      "start": 956,
      "end": 977
    },
    {
      "text": "the University/School",
      "label": "ORG",
      "start": 1236,
      "end": 1257
    },
    {
      "text

In [ ]:
DATA_PATH = Path("data/summarizerdata/final_dataset_augmented.jsonl")
OUTPUT_PATH = Path("data/ner_relationships.jsonl")

if not DATA_PATH.exists():
    candidates = [
        DATA_PATH,
        Path.cwd().parent / DATA_PATH,
        Path.cwd().parent.parent / DATA_PATH,
    ]

    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate.exists():
            DATA_PATH = candidate
            break
    else:
        raise FileNotFoundError(
            f"Dataset not found. Checked:\n"
            + "\n".join(f"  - {p.resolve()}" for p in candidates)
        )

print(f"Using dataset: {DATA_PATH}")

records = []
with DATA_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            print(f"Skipping invalid JSON on line {line_no}: {exc}")

print(f"Loaded {len(records)} records.")



Using dataset: C:\ProjectFiles\uni-comms-intelligence\data\summarizerdata\final_dataset_augmented.jsonl
Loaded 75 records.


In [8]:
def choose_source_text(record):
    """Prefer cleaned text; fall back to raw text."""
    text = record.get("cleaned_text") or record.get("raw_text") or ""
    return text if isinstance(text, str) else ""


processed = []

for i, record in enumerate(records, start=1):
    text = choose_source_text(record)
    extracted = extractor.extract_entities(text)

    output = {
        "notice_id": record.get("notice_id"),
        "document_id": record.get("document_id"),
        "title": record.get("title"),
        "category": record.get("category"),
        "university": record.get("university"),
        "extracted": extracted
    }
    processed.append(output)

    if i % 10 == 0 or i == len(records):
        print(f"Processed {i}/{len(records)}")

OUTPUT_PATH = DATA_PATH.parent.parent / "ner_relationships.jsonl"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for row in processed:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved: {OUTPUT_PATH}")


Processed 10/75
Processed 20/75
Processed 30/75
Processed 40/75
Processed 50/75
Processed 60/75
Processed 70/75
Processed 75/75
Saved: C:\ProjectFiles\uni-comms-intelligence\data\ner_relationships.jsonl


In [9]:
for row in processed:
    relations = row["extracted"]["relations"]
    if relations:
        print("NOTICE:", row["notice_id"])
        print("TITLE:", row["title"])
        print(json.dumps(relations[:10], indent=2, ensure_ascii=False))
        break
else:
    print("No relations found in the processed dataset.")


NOTICE: CIRC-2026-0029
TITLE: Statement dt: 23-02-2026
[
  {
    "type": "ACTION_INVOLVES_ORG",
    "source": "are requested to desist",
    "target": "Campus",
    "evidence": "All the stakeholders are requested to desist from indulging in any kind of unwarranted activity and cooperate in maintaining peace and harmony on the Campus failing which strict disciplinary action will be taken as per rules."
  }
]
